# 04a · Một neural network thực sự tính gì?

**Chỉ học mạng, chưa giải phương trình.** Sau bài này bạn giải thích được tensor shape `(N,1)`, `nn.Linear`, activation, `nn.Sequential`, tham số mạng và `model(x)`. Các ô chạy độc lập trên CPU. Đọc một ô Markdown rồi mới chạy ô code ngay sau nó.

## 1. Một tensor = nhiều điểm đầu vào

Mỗi hàng là một tọa độ $x_i$; một cột vì bài toán chỉ có một tọa độ. `dtype=torch.float64` dùng xuyên suốt các bài dầm.

In [ ]:
import torch
from torch import nn

x = torch.tensor([[0.0], [0.5], [1.0]], dtype=torch.float64)
print(x)
print("shape:", x.shape)     # 3 điểm, mỗi điểm có 1 số

## 2. Một lớp `Linear` là công thức $z=Wx+b$

`nn.Linear(1,1)` có một trọng số và một bias. PyTorch khởi tạo chúng; ta xem giá trị trước và đối chiếu `layer(x)` với phép tính tay. Ở đây **chưa có học**.

In [ ]:
torch.manual_seed(7)
layer = nn.Linear(1, 1).double()
z = layer(x)
z_by_hand = x @ layer.weight.T + layer.bias
print("weight:", layer.weight.item(), "bias:", layer.bias.item())
print("đầu ra:", z.reshape(-1).tolist())
print("khớp phép tính tay:", torch.allclose(z, z_by_hand))

## 3. Activation tạo phi tuyến

Sau phép affine, `Tanh` biến $z$ thành $a=\tanh(z)$. Nếu chỉ nối nhiều lớp Linear **mà không có activation**, toàn mạng vẫn là một hàm affine. PINN dầm thường dùng activation trơn vì cần đạo hàm bậc cao.

In [ ]:
a = torch.tanh(z)
print("trước Tanh:", z.reshape(-1).tolist())
print("sau Tanh:", a.reshape(-1).tolist())

## 4. Ghép thành mạng $u_\theta(x)$

`nn.Sequential` chạy các lớp **theo đúng thứ tự liệt kê**. `model(x)` là bước *forward*. `model.parameters()` chứa các trọng số/bias sẽ được học ở bài PINN; tọa độ $x$ không nằm trong danh sách này.

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 8),
    nn.Tanh(),
    nn.Linear(8, 1)
).double()
u = model(x)
print(model)
print("input shape:", x.shape, "output shape:", u.shape)
for name, value in model.named_parameters():
    print(name, tuple(value.shape))

**Tự làm:** thay `8` neuron thành `4`, chạy lại từ đầu, ghi shape của từng tham số. Đầu ra có còn shape `(3,1)` không? Giải thích vì sao. **Sản phẩm:** ảnh chụp đầu ra và 3–5 câu giải thích `Linear → Tanh → Linear`.

**Tiếp theo:** [04b · autograd](04b-autograd.ipynb). Khi đã hiểu mạng và đạo hàm, mới học vòng huấn luyện PINN.